# Dominio 4: Limpieza de Excel para Consultoría / Data Lake -> Data Warehouse

El World Development Indicators (WDI) completo del Banco Mundial: un Excel
real de ~80MB, 6 hojas, 401.394 filas país×indicador en la hoja `Data` -- el
tipo de archivo que una consultora recibe de un cliente y tiene que
transformar en un data warehouse consultable, no un CSV ya tabular.

Este notebook ejecuta el pipeline real de `src/domains/consulting_excel_dwh/`:
ingesta streaming del Excel completo -> filtrado a 10 indicadores curados ->
transformación a un warehouse DuckDB en esquema estrella (`dim_country`,
`dim_indicator`, `fact_indicator_value`) -> modelo de esperanza de vida.


## 1. De Excel a Data Warehouse

La hoja `Data` no se puede cargar completa a memoria en cada corrida (leerla
entera toma ~50s solo para iterar) -- `fetch.py` la recorre en streaming
(`openpyxl read_only=True`) y solo materializa los indicadores curados. La
hoja `Country` mezcla países reales con agregados regionales ("World", "OECD
members"...), distinguibles porque el campo `Region` está vacío en los
agregados.


In [ ]:
import sys
sys.path.insert(0, "..")
import json
import duckdb
import pandas as pd

report = json.load(open("../outputs/consulting/clean_report.json"))
for k, v in report.items():
    if not isinstance(v, list):
        print(f"{k}: {v}")


**Funnel de filas a través del ETL** (crudo ancho -> largo -> sin agregados -> tras interpolar):


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/etl_funnel.png"))


## 2. Consultando el warehouse


In [ ]:
con = duckdb.connect("../data/processed/consulting/wdi_warehouse.duckdb", read_only=True)
print(con.execute("SELECT COUNT(*) AS n_paises FROM dim_country").fetchdf())
print(con.execute("SELECT COUNT(*) AS n_filas_fact FROM fact_indicator_value").fetchdf())
con.execute("SELECT * FROM dim_indicator").fetchdf()


## 3. Features y modelado (>=100 épocas)


In [ ]:
from src.domains.consulting_excel_dwh.features import query_wide_panel, build_features

wide = query_wide_panel()
features_df = build_features(wide)
print(f"{len(features_df):,} filas x features, {features_df['country_code'].nunique()} países reales")
features_df.head()


In [ ]:
metrics = json.load(open("../outputs/consulting/metrics.json"))
pd.DataFrame(metrics["results"]).T


**Resultado real**: XGBoost (R²=0.938) y la MLP (R²=0.922) predicen la
esperanza de vida del año siguiente con alta precisión real a partir de gasto
en salud, agua/saneamiento, PIB per cápita y sus rezagos -- el baseline
(media histórica) tiene R² NEGATIVO (-1.61) porque la esperanza de vida tiene
una tendencia real al alza desde 1960 que un promedio histórico subestima
sistemáticamente en el período de test (2019-2024).


## 4. Gráficos


**% de nulos en el fact table, antes/después de interpolar por serie país×indicador.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/missingness_before_after.png"))


**Esperanza de vida real, 1960-2024: Chile vs. Haití vs. Japón** -- incluye historia real, no filtrada (ver nota abajo).


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/esperanza_vida_paises.png"))


**Correlación entre indicadores socioeconómicos y la esperanza de vida del año siguiente.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/feature_correlation.png"))


**Curva de entrenamiento de la MLP** -- >=100 épocas.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/mlp_training_curve.png"))


**Esperanza de vida real vs. predicha (XGBoost, holdout 2019-2024).**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/xgb_regression_diagnostics.png"))


**Comparación baseline vs. MLP vs. XGBoost.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/consulting/figures/model_comparison.png"))


## Conclusiones

- Un Excel real de 80MB no se limpia "a mano": requiere streaming, un modelo
  dimensional (esquema estrella) y una separación explícita entre países
  reales y agregados regionales.
- El dato real, sin filtrar, incluye eventos históricos extremos genuinos
  (Camboya 1976-78, Ruanda 1994: esperanza de vida ~11-12 años) -- se
  mantienen en el warehouse tal como el Banco Mundial los publica, no se
  recortan por "verse mal".
- El warehouse normalizado (`fact_indicator_value`, formato largo) es la capa
  reusable; el panel ancho para el modelo es una vista derivada específica
  de este caso de uso, no la tabla base.
